# Module 02 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_saxpy_buggy.cu](./adversarial_saxpy_buggy.cu) computes the
thread index with a multiplication instead of an addition:

```cpp
int tid = blockIdx.x * blockDim.x * threadIdx.x;   // WRONG
```

The standard global thread index is:

```cpp
int tid = blockIdx.x * blockDim.x + threadIdx.x;   // correct
```

With the buggy version, every thread whose `threadIdx.x == 0` gets `tid == 0`
(so they all pile onto element 0), and the other threads land on scattered,
mostly out-of-range indices that the `if (tid < N)` guard then skips. The result
is that the vast majority of `c[]` is never written — it keeps its initial value.

**Why it fools a quick check:** the original program only prints `c[0..4]` and
`c[N-5..N-1]`. Element 0 *is* written (many times), so `c[0]` looks correct, and
the tail may coincidentally print something. Inspecting 10 of 4 million elements
is not a test.

The fix is a single character: `*` → `+`. The corrected, prefetching version
with a full-array gate is [saxpy_solution.cu](solutions/saxpy_solution.cu).

## The unified-memory / prefetch lesson

- **First run, no prefetch:** the arrays are first touched on the host (the init
  loop), so their pages live in host memory. When the kernel runs, the GPU faults
  each page in on demand — visible in `nsys` as many "Unified Memory" page-fault
  events. This can dominate the runtime.
- **With `cudaMemPrefetchAsync`:** the pages migrate to the device up front in
  large transfers, so the kernel finds them resident. The page-fault storm
  disappears and the kernel time reflects actual compute + bandwidth.

SAXPY moves 3 integers and does ~2 operations per element, so it is firmly
**memory-bound**. Once page faults are removed, its speed is set by memory
bandwidth, not arithmetic — that is the ceiling to reason about in Phase 5.

## Mapping to the 5-phase loop

- **PREDICT:** reading the kernel, you should flag the index expression as a
  correctness risk *before* running.
- **VERIFY:** a full-array comparison against `saxpy_cpu` turns the silent index
  bug into a reliable `FAIL`. Printing a handful of values does not.
- **DIAGNOSE:** `nsys` shows the page-fault behaviour and whether prefetch moved
  the bottleneck — the evidence your Phase-5 write-up needs.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!nvcc -O3 -arch=native solutions/saxpy_solution.cu -o sol && ./sol